In [ ]:
#Connectin to Kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
TOKEN = {"username":"*********","key":"*************"}
!pip install kaggle==1.5.12
!mkdir -p .kaggle
!mkdir -p /content & mkdir -p /content/.kaggle & mkdir -p /root/.kaggle/

with open('/content/.kaggle/kaggle.json', 'w') as file:
    json.dump(TOKEN, file)
!pip install --upgrade --force-reinstall --no-deps kaggle
!ls "/content/.kaggle"
!chmod 600 /content/.kaggle/kaggle.json
!cp /content/.kaggle/kaggle.json /root/.kaggle/

!kaggle config set -n path -v /content

  Using cached kaggle-1.5.12-py3-none-any.whl
  Attempting uninstall: kaggle
    Found existing installation: kaggle 1.5.12
    Uninstalling kaggle-1.5.12:
      Successfully uninstalled kaggle-1.5.12
kaggle.json
- path is now set to: /content


In [ ]:
!kaggle competitions download -c 11-785-s22-hw1p2

In [ ]:
!unzip /content/competitions/11-785-s22-hw1p2/11-785-s22-hw1p2.zip -d /content/

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

#!cp "/content/drive/MyDrive/HW1P2/11-785-s22-hw1p2.zip" "/content"
#!cp "/content/drive/MyDrive/HW1P2/train_filenames_subset_0008_v2.csv" "/content"

#!cp "/content/drive/MyDrive/HW1P2/train_filenames_subset_8192_v2.csv" "/content"

#!cp "/content/drive/MyDrive/HW1P2/train_filenames_subset_0512_v2.csv" "/content"

#!cp "/content/drive/MyDrive/HW1P2/train_filenames_subset_2048_v2.csv" "/content"

#!cp -r "/content/drive/MyDrive/HW1P2/hw1p2_student_data" "/content"

#!unzip /content/drive/MyDrive/HW1P2/11-785-s22-hw1p2.zip -d  /content/


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!mkdir /content/drive/MyDrive/HW1P2

In [ ]:
import os
import csv
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import pandas as pd




In [ ]:
from torch.nn.modules.linear import Linear
class Network(torch.nn.Module):
    def __init__(self, args):
        super(Network, self).__init__()
        # TODO: Please try different architectures
        in_size = 13 * (args["context"] * 2 + 1 )
        layers = [
            nn.Linear(in_size, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(4096, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 200),
            nn.BatchNorm1d(200),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(200, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 40)
        ]
        self.laysers = nn.Sequential(*layers)

    def forward(self, A0):
        x = self.laysers(A0)
        return x

In [ ]:
class LibriSamples(torch.utils.data.Dataset):
    def __init__(self, data_path, sample=20000, shuffle=True, partition="dev-clean", csvpath=None):
        # sample represent how many npy files will be preloaded for one __getitem__ call
        self.sample = sample
        #self.Y_names = None

        # using a small part of the dataset to debug
        if csvpath:
          if partition == 'train-clean-100':
              self.X_dir = data_path + "/" + partition + "/mfcc/"
              self.Y_dir = data_path + "/" + partition +"/transcript/"
              self.X_names = os.listdir(self.X_dir)
              self.Y_names = os.listdir(self.Y_dir)

              subset = self.parse_csv(csvpath)
              self.X_names = [i for i in self.X_names if i in subset]
              self.Y_names = [i for i in self.Y_names if i in subset]
          else:
              self.X_dir = data_path + "/" + partition + "/mfcc/"
              self.X_names = list(pd.read_csv(csvpath).file)
              self.Y_names = None
        else:
          self.X_dir = data_path + "/" + partition + "/mfcc/"
          self.Y_dir = data_path + "/" + partition +"/transcript/"
          self.X_names = os.listdir(self.X_dir)
          self.Y_names = os.listdir(self.Y_dir)
        if shuffle == True:
          if self.Y_names != None:
            XY_names = list(zip(self.X_names, self.Y_names))
            random.shuffle(XY_names)
            self.X_names, self.Y_names = zip(*XY_names)

         # assert(len(self.X_names) == len(self.Y_names))

        self.length = len(self.X_names)

        self.PHONEMES = [
            'SIL',   'AA',    'AE',    'AH',    'AO',    'AW',    'AY',
            'B',     'CH',    'D',     'DH',    'EH',    'ER',    'EY',
            'F',     'G',     'HH',    'IH',    'IY',    'JH',    'K',
            'L',     'M',     'N',     'NG',    'OW',    'OY',    'P',
            'R',     'S',     'SH',    'T',     'TH',    'UH',    'UW',
            'V',     'W',     'Y',     'Z',     'ZH',    '<sos>', '<eos>']

    @staticmethod
    def parse_csv(filepath):
        subset = []
        with open(filepath) as f:
            f_csv = csv.reader(f)
            for row in f_csv:
                subset.append(row[1])
        return subset[1:]

    def __len__(self):
        return int(np.ceil(self.length / self.sample))

    def __getitem__(self, i):
        sample_range = range(i*self.sample, min((i+1)*self.sample, self.length))

        X, Y = [], []
        if self.Y_names != None:
          for j in sample_range:
              X_path = self.X_dir + self.X_names[j]
              Y_path = self.Y_dir + self.Y_names[j]

              label = [self.PHONEMES.index(yy) for yy in np.load(Y_path)][1:-1]

              X_data = np.load(X_path)
              X_data = (X_data - X_data.mean(axis=0))/X_data.std(axis=0)
              X.append(X_data)
              Y.append(np.array(label))

          X, Y = np.concatenate(X), np.concatenate(Y)
          return X, Y

        else:
            for j in sample_range:
              X_path = self.X_dir + self.X_names[j]

              X_data = np.load(X_path)
              X_data = (X_data - X_data.mean(axis=0))/X_data.std(axis=0)
              X.append(X_data)

            X = np.concatenate(X)
            return X


class LibriItems(torch.utils.data.Dataset):
    def __init__(self, X, Y, context = 0):

      if Y is not None:
        assert(X.shape[0] == Y.shape[0])

      self.length  = X.shape[0]
      self.context = context

      if context == 0:
          self.X, self.Y = X, Y
      else:
          X = np.pad(X, ((context,context), (0,0)), 'constant', constant_values=(0,0))
          self.X, self.Y = X, Y

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        if self.Y is  None:
          if self.context == 0:
              xx = self.X[i].flatten()
          else:
              xx = self.X[i:(i + 2*self.context + 1)].flatten()
          return xx
        else:
          if self.context == 0:
              xx = self.X[i].flatten()
              yy = self.Y[i]
          else:
              xx = self.X[i:(i + 2*self.context + 1)].flatten()
              yy = self.Y[i]
          return xx, yy




In [ ]:
def train(args, model, device, train_samples, optimizer, criterion, epoch):
    model.train()
    for i in range(len(train_samples)):
        X, Y = train_samples[i]
        train_items = LibriItems(X, Y, context=args['context'])
        train_loader = torch.utils.data.DataLoader(train_items, batch_size=args['batch_size'], shuffle=True)

        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.float().to(device)
            target = target.long().to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % args['log_interval'] == 0:
                print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    epoch, batch_idx * len(data), len(train_loader.dataset),
                    100. * batch_idx / len(train_loader), loss.item()))


def test(args, model, device, dev_samples, criterion):
    model.eval()
    true_y_list = []
    pred_y_list = []
    allLosses = []
    with torch.no_grad():
        for i in range(len(dev_samples)):
            X, Y = dev_samples[i]

            test_items = LibriItems(X, Y, context=args['context'])
            test_loader = torch.utils.data.DataLoader(test_items, batch_size=args['batch_size'], shuffle=False)

            for data, true_y in test_loader:
                data = data.float().to(device)
                true_y = true_y.long().to(device)

                output = model(data)
                loss = criterion(output, true_y)
                allLosses.append(loss.cpu().numpy().item())
                pred_y = torch.argmax(output, axis=1)

                pred_y_list.extend(pred_y.tolist())
                true_y_list.extend(true_y.tolist())

    train_accuracy =  accuracy_score(true_y_list, pred_y_list)
    return train_accuracy, np.array(allLosses).mean()


def predict(args, model, device):
   model.eval()
   pred_y_list = []

   dev_samples = LibriSamples(
       data_path = args['LIBRI_PATH'],
       shuffle=False,
       partition="test-clean",
       csvpath="test_order.csv"
      )
   with torch.no_grad():
        for i in range(len(dev_samples)):
            X = dev_samples[i]

            test_items = LibriItems(X, None, context=args['context'])
            test_loader = torch.utils.data.DataLoader(
                test_items, batch_size=args['batch_size'], shuffle=False)

            for data in test_loader:
                data = data.float().to(device)
                # true_y = true_y.long().to(device)

                output = model(data)
                pred_y = torch.argmax(output, axis=1)

                pred_y_list.extend(pred_y.tolist())

   return pred_y_list



def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = Network(args).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args['lr'])
    criterion = torch.nn.CrossEntropyLoss()
    # If you want to use full Dataset, please pass None to csvpath
    train_samples = LibriSamples(
        data_path = args['LIBRI_PATH'],
        shuffle=True, partition="train-clean-100",
        # csvpath="/content/drive/MyDrive/HW1P2/train_filenames_subset_0008_v2.csv"
      )

    dev_samples = LibriSamples(data_path = args['LIBRI_PATH'], shuffle=True, partition="dev-clean")

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1, factor=0.5)

    for epoch in range(1, args['epoch'] + 1):
        train(args, model, device, train_samples, optimizer, criterion, epoch)
        test_acc, test_loss = test(args, model, device, dev_samples, criterion)

        print('Dev accuracy ', test_acc)
        torch.save(model, "/content/drive/MyDrive/HW1P2/model.bin")
        scheduler.step(test_loss)

    prediction = predict(args, model, device)

    import pandas as pd
    pred = pd.DataFrame({"id": range(0, len(prediction)), "label": prediction})
    pred.to_csv("submission.csv", index=False)

    return model


if __name__ == '__main__':
    args = {
        'batch_size': 4096,
        'context': 30,
        'log_interval': 200,
        'LIBRI_PATH': '/content/hw1p2_student_data',
        'lr': 0.001,
        'epoch': 20
    }
    model = main(args)


Train Epoch: 1 [0/25372101 (0%)]	Loss: 3.840484
Train Epoch: 1 [819200/25372101 (3%)]	Loss: 1.485809
Train Epoch: 1 [1638400/25372101 (6%)]	Loss: 1.319021
Train Epoch: 1 [2457600/25372101 (10%)]	Loss: 1.180242
Train Epoch: 1 [3276800/25372101 (13%)]	Loss: 1.102013
Train Epoch: 1 [4096000/25372101 (16%)]	Loss: 1.060404
Train Epoch: 1 [4915200/25372101 (19%)]	Loss: 1.032580
Train Epoch: 1 [5734400/25372101 (23%)]	Loss: 0.999633
Train Epoch: 1 [6553600/25372101 (26%)]	Loss: 1.009228
Train Epoch: 1 [7372800/25372101 (29%)]	Loss: 0.991075
Train Epoch: 1 [8192000/25372101 (32%)]	Loss: 0.990968
Train Epoch: 1 [9011200/25372101 (36%)]	Loss: 0.956880
Train Epoch: 1 [9830400/25372101 (39%)]	Loss: 0.989993
Train Epoch: 1 [10649600/25372101 (42%)]	Loss: 0.939472
Train Epoch: 1 [11468800/25372101 (45%)]	Loss: 0.920500
Train Epoch: 1 [12288000/25372101 (48%)]	Loss: 0.923649
Train Epoch: 1 [13107200/25372101 (52%)]	Loss: 0.879348
Train Epoch: 1 [13926400/25372101 (55%)]	Loss: 0.841531
Train Epoch: 1 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load("/content/drive/MyDrive/HW1P2/model.bin")
prediction = predict(args, model, device)
pred = pd.DataFrame({"id": range(0, len(prediction)), "label": prediction})
pred.to_csv("submission.csv", index=False)

In [ ]:
pd.read_csv("submission.csv").shape

(1943253, 2)

In [ ]:
!kaggle competitions submit -c 11-785-s22-hw1p2 -f submission.csv -m "Message"

100% 18.6M/18.6M [00:00<00:00, 29.9MB/s]
Successfully submitted to Frame-Level Speech Recognition